# Air Quality Analysis and Carbon Monoxide Concentration Prediction Using Exploratory Data Analysis and Deep Learning

**Course Code:** U21ADP05 — Exploratory Data Analysis and Visualization  
**Assignment:** Assignment II — Case Study  
**Student Name:** Kavin M  
**Roll Number:** 24AD053  
**Degree & Department:** B.Tech. Artificial Intelligence and Data Science (III Year / V Semester)  
**Institution:** KPR Institute of Engineering and Technology  
**Academic Year:** 2026–2027 (Odd Semester)  
**Course Coordinator:** Mr. Rushikesh Kadam  
**Dataset:** UCI Machine Learning Repository — Air Quality Dataset (DOI: [10.24432/C59K5F](https://doi.org/10.24432/C59K5F))  


In [ ]:
# 1. Imports and Reproducibility Setup
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import set_seed, ensure_directories, calculate_regression_metrics
from src.data_loader import load_raw_dataset, inspect_data_quality
from src.preprocessing import clean_air_quality_dataframe, prepare_modeling_data, RAW_PREDICTOR_COLS, TARGET_COL
from src.train_model import build_baseline_mlp, build_deep_mlp

set_seed(42)
dirs = ensure_directories(project_root)
print('TensorFlow Version:', tf.__version__)
print('Project Root:', project_root)


## 2. Dataset Acquisition and Initial Data Quality Inspection
We load the official UCI Air Quality dataset (9,358 hourly records, 15 attributes).
The raw CSV file uses semicolons as delimiters, commas as European decimal separators, and represents missing sensor readings as -200.


In [ ]:
# Load raw dataset
raw_df = load_raw_dataset()
print('Raw dataset shape:', raw_df.shape)
raw_df.head(5)


In [ ]:
# Inspect Data Quality and Missing Sentinel Counts
quality_summary = inspect_data_quality(raw_df)
print('Total Valid Records:', quality_summary['total_valid_rows'])
print('Target Variable:', quality_summary['target_variable'])
print('Target Missing (-200 sentinel count):', quality_summary['target_missing_count'])
print('Target Usable Observations:', quality_summary['target_valid_count'])

summary_df = pd.read_csv(dirs['tables'] / 'data_quality_summary.csv')
summary_df


## 3. Data Cleaning, Target Leakage Prevention & Feature Engineering
1. **Sentinel Conversion:** Convert -200 to NaN.
2. **Date/Time Parsing:** Combine Date (DD/MM/YYYY) and Time (HH.MM.SS) into a single continuous DateTime timestamp.
3. **Target Exclusions:** Rows where CO(GT) is missing (1,683 rows) are discarded from the supervised learning set.
4. **Target Leakage Prevention:** All ground-truth pollutant concentrations (CO(GT), NMHC(GT), C6H6(GT), NOx(GT), NO2(GT)) are excluded from model inputs. Inputs comprise only low-cost chemical sensor responses (PT08.S1 to PT08.S5), weather readings (T, RH, AH), and engineered cyclical time features.
5. **Feature Engineering:** Sine and cosine encodings for diurnal hour (24h) and seasonal month (12m) cycles.
6. **Chronological Splitting:** 70% Train (5,371 rows), 15% Validation (1,151 rows), 15% Test (1,152 rows).
7. **Train-Only Scaling & Imputation:** Imputer (median) and StandardScaler fit strictly on the training partition.


In [ ]:
clean_df = clean_air_quality_dataframe(raw_df)
prep_data = prepare_modeling_data(clean_df)

X_train, y_train = prep_data['X_train'], prep_data['y_train']
X_val, y_val = prep_data['X_val'], prep_data['y_val']
X_test, y_test = prep_data['X_test'], prep_data['y_test']
feature_names = prep_data['feature_names']

print('Feature Count:', len(feature_names))
print('Features:', feature_names)
print('X_train shape:', X_train.shape, '| y_train shape:', y_train.shape)
print('X_val shape:  ', X_val.shape, '| y_val shape:  ', y_val.shape)
print('X_test shape: ', X_test.shape, '| y_test shape: ', y_test.shape)


## 4. Exploratory Data Analysis (EDA)
We display the generated publication-grade EDA visualizations.


In [ ]:
# Descriptive statistics summary table
desc_df = pd.read_csv(dirs['tables'] / 'descriptive_statistics.csv', index_col=0)
desc_df


In [ ]:
# Display Key Generated Figures
from IPython.display import Image, display

eda_figs = [
    'fig01_missing_values.png',
    'fig02_target_distribution.png',
    'fig03_sensor_distributions.png',
    'fig04_boxplots_outliers.png',
    'fig05_correlation_heatmap.png',
    'fig06_time_series.png',
    'fig07_bivariate_scatter.png',
    'fig08_multivariate_scatter.png',
    'fig09_diurnal_hourly_profile.png'
]

for f in eda_figs:
    p = dirs['figures_eda'] / f
    if p.exists():
        display(Image(filename=str(p)))


## 5. Competing Deep Learning MLP Regression Architectures
We compare two Multi-Layer Perceptron architectures on the chronological validation set:
- **Model A (Baseline MLP):** Dense(64, ReLU) -> Dropout(0.10) -> Dense(32, ReLU) -> Linear
- **Model B (Deep Regularized MLP):** Dense(128, ReLU) -> Dropout(0.20) -> Dense(64, ReLU) -> Dropout(0.15) -> Dense(32, ReLU) -> Linear
Model selection is based strictly on validation MAE.


In [ ]:
# Display Model Comparison Table
model_comp_df = pd.read_csv(dirs['tables'] / 'model_comparison.csv')
model_comp_df


## 6. Final Evaluation on Held-Out Test Data
Evaluating the selected model (Model B) once on the chronological test partition (1,152 unseen observations).


In [ ]:
with open(dirs['root'] / 'outputs' / 'metrics.json', 'r') as f:
    test_metrics = json.load(f)

for k, v in test_metrics.items():
    print(f'{k:12s}: {v}')


In [ ]:
# Display Evaluation Diagnostic Figures
eval_figs = [
    'fig10_training_loss_curve.png',
    'fig11_predicted_vs_actual.png',
    'fig12_residual_distribution.png',
    'fig13_residuals_vs_predicted.png',
    'fig14_feature_importance.png',
    'fig15_test_chronological_series.png'
]

for f in eval_figs:
    p = dirs['figures_eval'] / f
    if p.exists():
        display(Image(filename=str(p)))


## 7. Permutation Feature Importance Analysis
We examine feature importance by measuring validation MAE degradation when each feature is permuted across 5 repeats.


In [ ]:
imp_df = pd.read_csv(dirs['tables'] / 'feature_importance.csv')
imp_df


## 8. Summary and Conclusions
- **Prediction Target:** Carbon Monoxide concentration CO(GT) in mg/m³.
- **Validation-Selected Model:** Model B (Deep Regularized MLP with 3 hidden layers: 128 -> 64 -> 32 units, Dropout).
- **Test Performance:** Achieved R² = 0.8094, MAE = 0.3658 mg/m³, and RMSE = 0.5564 mg/m³ on the held-out test partition.
- **Key Predictors:** Sensor responses PT08.S4(NO2) and PT08.S2(NMHC), alongside Absolute Humidity (AH) and PT08.S1(CO), provide the strongest predictive signals.
